# Notebook 7: Advanced SBI Methods

In the previous notebooks, life was easy: the data vector had only 10 entries, and we could run the simulator for any parameters we liked. Real applications, for example in large-scale structure cosmology, often differ in two ways:

1. **The data vector is large.** Density estimators work best in low dimensions, so the data have to be compressed to a few informative summary statistics first.
2. **The simulations do not follow the prior.** Simulations are expensive, and existing simulation suites were often designed for a different purpose (for example with a denser sampling around a fiducial model). NPE, however, implicitly uses the distribution of the training parameters as its prior, which makes it unfeasible in this situation.

In this notebook, we will use a 2-step process which deals with both problems: we first find an optimal summary network through an embedding network using VMIM, and then train a second estimator on the likelihood of the compressed data.

We will also use a more flexible approach for training and MCMC sampling, allowing for greater customization: we only take the density estimators from the `sbi` package and write the training loop, the MCMC sampling and the coverage test ourselves. Notebook 8 then runs the same pipeline with the `sbikit` package, which provides these steps ready-made.

**What you will learn**
- how an embedding network learns summary statistics (VMIM)
- why NPE gives wrong posteriors if the training simulations do not follow the prior, and how to detect this
- how NLE on the compressed data avoids the problem
- how to write your own training loop for `sbi` density estimators and how to sample with `emcee`
- how to check the calibration of the result and how much information the compression lost

**Runtime:** about three minutes on a laptop.

## Methods

We will explore a technique called VMIM (Variational Mutual Information Maximization) using the methods implemented in the `sbi` package:
1.  **NPE of raw data**: Trains an embedding network using the VMIM loss.
2.  **NLE**: Learn likelihood of compressed data and then use MCMC to sample from the posterior.

## Task: Log-Normal Toy Model

We will infer parameters in a log-normal toy model.

In [ ]:
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional
from copy import deepcopy
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sbi.neural_nets.net_builders import build_nsf, build_mdn
from sbi.inference import likelihood_estimator_based_potential
from sbi.inference.posteriors import DirectPosterior
from sbi.utils import BoxUniform

import emcee
from chainconsumer import ChainConsumer, Chain, Truth, PlotConfig
from tarp import get_tarp_coverage

# The toy problem of this notebook (shared with Notebook 8)
from tutorial_utils import model_fn, generate_data, lognormal_benchmark, exact_log_posterior

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Our models are small, so they do not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## Create Simulations

Our simulator has three parameters $\theta = (\theta_1, \theta_2, \theta_3)$ and produces a data vector with 30 entries, $x = \mu(\theta) + n$:
- The model $\mu(\theta)$ consists of three blocks of ten entries, which are proportional to $\theta_1$, $\theta_2^2$ and $\theta_3^3$, respectively.
- The noise $n$ is log-normal (with slightly correlated entries), shifted such that it has zero mean. It is strongly skewed.

Note that the data only depend on $\theta_2^2$. The sign of $\theta_2$ can therefore not be inferred and its posterior is symmetric around zero (and bimodal if $|\theta_2|$ is large enough).

In [ ]:
# The simulator lives in tutorial_utils.py, because Notebook 8 uses it as well. Here is its source code:
print(inspect.getsource(model_fn))
print(inspect.getsource(generate_data))

In [ ]:
# A fixed set of simulations (the same in Notebook 8). lognormal_benchmark() calls the simulator like this:
#
#   x_train_npe, theta_train_npe = generate_data(10000, grid='combined')     # training data doesn't follow the prior
#   x_test, theta_test = generate_data(1000, grid='wide')                    # test data follows the prior
#   x_fiducial, theta_fiducial = generate_data(100, np.zeros((100, 3)))      # fiducial model: theta = [0, 0, 0]
benchmark = lognormal_benchmark()

# Training data doesn't follow Prior
x_train_npe, theta_train_npe = benchmark.x_train_npe, benchmark.theta_train_npe
x_val_npe, theta_val_npe = benchmark.x_val_npe, benchmark.theta_val_npe
x_train_nle, theta_train_nle = benchmark.x_train_nle, benchmark.theta_train_nle
x_val_nle, theta_val_nle = benchmark.x_val_nle, benchmark.theta_val_nle

# Test data follows Prior
x_test, theta_test = benchmark.x_test, benchmark.theta_test

# Our "observation" is a single realization at the fiducial model.
# Note: we do NOT use the mean of the fiducial realizations as observation. Because the noise is strongly
# skewed, such an averaged ("noise-free") data vector does not look like a typical realization of our
# simulator, and analysing it with a likelihood (or posterior) learned for single noisy realizations
# leads to a biased result. (For Gaussian noise, using the mean would be fine.)
x_obs, theta_obs = benchmark.x_obs, benchmark.theta_obs

print("Training data for the compression:", x_train_npe.shape, theta_train_npe.shape)
print("Training data for NLE:            ", x_train_nle.shape, theta_train_nle.shape)
print("Test data:                        ", x_test.shape, theta_test.shape)

In [ ]:
prior = BoxUniform(-2*torch.ones(3), 2*torch.ones(3))

### The Training Simulations do not Follow the Prior

Our prior is uniform in the cube $[-2, 2]^3$, and the test simulations are drawn from it. The training simulations were generated differently (`grid='combined'`): half of them cover the prior range, the other half is concentrated in the smaller cube $[-1, 1]^3$. This mimics a simulation suite with a denser sampling around a fiducial model. Inside the small cube, the density of training simulations is nine times higher than outside.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))

axs[0].scatter(theta_train_npe[:, 0], theta_train_npe[:, 1], s=1, alpha=0.3)
axs[0].plot([-2, 2, 2, -2, -2], [-2, -2, 2, 2, -2], color='black', label='Prior range')
axs[0].set(xlabel=r'$\theta_1$', ylabel=r'$\theta_2$', title='Parameters of the training simulations')
axs[0].legend(loc='upper right')

bins = np.linspace(-2, 2, 41)
axs[1].hist(theta_train_npe[:, 0].numpy(), bins=bins, density=True, alpha=0.6, label='Training simulations')
axs[1].hist(prior.sample((10000,))[:, 0].numpy(), bins=bins, density=True, histtype='step', color='black', label='Samples from the prior')
axs[1].set(xlabel=r'$\theta_1$', ylabel='Density', title=r'Distribution of $\theta_1$')
axs[1].legend()
plt.show()

## NPE with embedding network

The embedding network $s_\phi$ compresses the 30-dimensional data vector to 6 summary statistics, $s = s_\phi(x)$. On top of it, we use a simple Mixture Density Network (with a single Gaussian component) as posterior estimator $q(\theta|s)$. Both are trained together by minimizing

$$ \mathcal{L} = -\mathbb{E}_{(\theta, x)} \left[ \log q(\theta \,|\, s_\phi(x)) \right] $$

This is ordinary NPE, with one additional network in front. It is also the **VMIM** loss. The name comes from the mutual information $I(\theta; s)$ between the parameters and the summaries, which measures how much we learn about $\theta$ from $s$. For any density estimator $q$:

$$ I(\theta; s) = H(\theta) - H(\theta|s) \;\geq\; H(\theta) + \mathbb{E} \left[ \log q(\theta|s) \right] = H(\theta) - \mathcal{L} $$

The entropy $H(\theta)$ of the training parameters is a constant. Minimizing $\mathcal{L}$ therefore maximizes a lower bound on the mutual information: the embedding network learns the summary statistics that are most informative about $\theta$.

In [ ]:
class FCNet(nn.Module):

    def __init__(
        self,
        input_dim: int = 1440,
        hidden_features: int = 1024,
        output_dim: int = 18,
        num_blocks: int = 2,
        dropout: float = 0.1,
        norm_type: Optional[str] = 'layer',
        activation: type[nn.Module] = nn.ReLU
    ) -> None:

        super().__init__()

        self.block_list = nn.ModuleList()

        if norm_type == 'layer':
            self.norm = nn.LayerNorm(input_dim)
        elif norm_type == 'batch':
            self.norm = nn.BatchNorm1d(input_dim)
        elif norm_type is not None:
            raise ValueError(f"Unsupported norm_type: {norm_type}")

        for i in range(num_blocks):
            in_features = input_dim if i == 0 else hidden_features
            block = nn.Sequential(
                nn.Linear(in_features, hidden_features),
                activation(inplace=True),
                nn.Dropout(dropout),
            )
            self.block_list.append(block)

        self.head = nn.Linear(hidden_features, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if hasattr(self, "norm"):
            x = self.norm(x)
        for block in self.block_list:
            x = block(x)
        return self.head(x)

In [ ]:
def build_npe(theta_train, x_train, num_components=1):
    # norm_type=None: sbi already standardizes every entry of the data vector (z_score_y below).
    # A LayerNorm on the input would in addition normalize every data vector by its own mean and
    # standard deviation. This removes the overall amplitude of the signal, and with it most of
    # the information about the parameters (try norm_type='layer' and compare the validation loss).
    embedding_net = FCNet(input_dim=30, hidden_features=32, num_blocks=2,  output_dim=6, dropout=0.1, norm_type=None)

    # Note the order of the arguments: the density estimator models p(first argument | second argument)
    return build_mdn(theta_train,
                     x_train,
                     z_score_x='independent',
                     z_score_y='independent',
                     hidden_features=64,
                     num_components=num_components,
                     embedding_net=embedding_net)

def make_loaders(inputs_train, condition_train, inputs_val, condition_val):
    # Each batch consists of (input, condition) of the density estimator
    dataset_training = TensorDataset(inputs_train, condition_train)
    dataset_validation = TensorDataset(inputs_val, condition_val)

    train_loader_kwargs = {
        "batch_size": 200,
        "drop_last": True,
        "shuffle": True,
    }

    val_loader_kwargs = {
        "batch_size": 500,
        "shuffle": False,
    }

    return DataLoader(dataset_training, **train_loader_kwargs), DataLoader(dataset_validation, **val_loader_kwargs)

density_estimator_npe = build_npe(theta_train_npe, x_train_npe)
train_loader_npe, val_loader_npe = make_loaders(theta_train_npe, x_train_npe, theta_val_npe, x_val_npe)

### Now the Training Loop

We write the training loop ourselves: Adam optimizer, a learning rate scheduler which reduces the learning rate when the validation loss stops improving, and early stopping. Since we need the same loop again for NLE below, we put it into a function.

In [ ]:
def train_density_estimator(density_estimator, train_loader, val_loader, lr=1e-3, max_num_epochs=1000, stop_after_epochs=25, name=''):

    optimizer = Adam(list(density_estimator.parameters()), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

    epoch = 0
    best_model_state_dict = {}
    epochs_since_last_improvement = 0
    best_val_loss = float("Inf")
    converged = False

    while epoch < max_num_epochs and not converged:

        print(f"Training {name} epoch {epoch}...", end='\r')

        density_estimator.train()
        for inputs_batch, condition_batch in train_loader:
            optimizer.zero_grad()

            # The loss is the negative log-probability of the inputs given the condition
            train_losses = density_estimator.loss(inputs_batch, condition=condition_batch)
            train_loss = torch.mean(train_losses)

            train_loss.backward()

            optimizer.step()

        epoch += 1

        # Calculate validation performance.
        density_estimator.eval()
        val_loss_sum = 0
        num_val = 0

        with torch.no_grad():
            for inputs_batch, condition_batch in val_loader:
                val_losses = density_estimator.loss(inputs_batch, condition=condition_batch)
                val_loss_sum += val_losses.sum().item()
                num_val += len(val_losses)

        # Take mean over all validation samples.
        val_loss = val_loss_sum / num_val

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_since_last_improvement = 0
            best_model_state_dict = deepcopy(density_estimator.state_dict())
        else:
            epochs_since_last_improvement += 1

        # If no validation improvement over many epochs, stop training.
        if epochs_since_last_improvement > stop_after_epochs:
            converged = True
            print(f'{name} successfully converged after {epoch} epochs')

    if not converged:
        print(f"{name} reached maximum number of epochs: {max_num_epochs}")

    # Restore the best model
    density_estimator.load_state_dict(best_model_state_dict)
    print(f'Best validation loss: {best_val_loss:.4f}')

    # Make sure model is in eval mode after training is finished
    density_estimator.eval()


train_density_estimator(density_estimator_npe, train_loader_npe, val_loader_npe, lr=1e-3, name='NPE')

## Can we Trust this NPE Posterior?

Training an embedding network with the VMIM loss is NPE, so we already have a posterior estimator. Before we use it, we check its calibration with the TARP coverage test from Notebook 4. The test simulations are drawn from the prior, as it should be. Sampling from the NPE posterior is fast, so the test only takes a few seconds.

In [ ]:
# Test set: drawn from the prior (unlike the training set)
# We use a subset of the test sims (using all 1000 takes longer, but gives smaller error bars)
num_test = 300
theta_tarp = theta_test[:num_test]
x_tarp = x_test[:num_test]

def sample_npe(density_estimator, num_samples=2500):
    """NPE posterior samples for all test observations: [num_samples, num_test, num_params]"""
    # DirectPosterior samples from the density estimator and rejects samples outside of the prior range
    posterior = DirectPosterior(density_estimator, prior)
    samples = [posterior.sample((num_samples,), x=x_tarp[i], show_progress_bars=False) for i in range(num_test)]
    return torch.stack(samples, dim=1)

def tarp_coverage(posterior_samples):
    # TARP expects the samples as [num_samples, num_test, num_params] and the true parameters as [num_test, num_params]
    # norm=True rescales the parameters to the unit hypercube, from which the random reference points are drawn
    # bootstrap=True repeats the test for resampled test sets, which gives us an error estimate
    ecp, alpha = get_tarp_coverage(posterior_samples.numpy(), theta_tarp.numpy(), references='random', metric='euclidean',
                                   norm=True, num_alpha_bins=20, num_bootstrap=100, bootstrap=True)
    return alpha, ecp.mean(axis=0), ecp.std(axis=0)

def plot_coverage(coverage_results):
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], 'k--', label='Ideal')
    for name, (alpha, ecp_mean, ecp_std) in coverage_results.items():
        plt.errorbar(alpha, ecp_mean, yerr=ecp_std, label=name)
        # Largest deviation from the diagonal, also in units of its (bootstrap) error
        deviation = np.abs(ecp_mean - alpha)
        i = deviation.argmax()
        print(f"{name.replace(chr(10), ' ')}: largest deviation from the diagonal = {deviation[i]:.2f} ({deviation[i] / max(ecp_std[i], 1e-12):.1f} sigma)")
    plt.xlabel('Credibility Level')
    plt.ylabel('Expected Coverage')
    plt.title('TARP Coverage Test')
    plt.legend()
    plt.show()

coverage_results = {'NPE (1 component)': tarp_coverage(sample_npe(density_estimator_npe))}
plot_coverage(coverage_results)

The curve is far away from the diagonal: these posteriors cannot be trusted. There are two suspects:

1. **The posterior estimator is too simple.** A single Gaussian cannot represent a posterior with two peaks or with sharp edges.
2. **The training simulations do not follow the prior.**

To find out, we train two more estimators: one with a more flexible posterior (a mixture of five Gaussians) on the same simulations, and one with the same flexible posterior on new simulations which do follow the prior. The second one is a control experiment which is only possible because our simulator is cheap.

In [ ]:
# Suspect 1: a more flexible posterior estimator (mixture of 5 Gaussians), same training simulations
density_estimator_npe_flexible = build_npe(theta_train_npe, x_train_npe, num_components=5)
train_density_estimator(density_estimator_npe_flexible, train_loader_npe, val_loader_npe, lr=1e-3, name='NPE (5 components)')

# Suspect 2: the same flexible estimator, but trained on simulations which follow the prior
x_train_prior, theta_train_prior = generate_data(10000, grid='wide')
x_val_prior, theta_val_prior = generate_data(1000, grid='wide')
density_estimator_npe_prior = build_npe(theta_train_prior, x_train_prior, num_components=5)
train_loader_prior, val_loader_prior = make_loaders(theta_train_prior, x_train_prior, theta_val_prior, x_val_prior)
train_density_estimator(density_estimator_npe_prior, train_loader_prior, val_loader_prior, lr=1e-3, name='NPE (5 components, simulations from prior)')

coverage_results['NPE (5 components)'] = tarp_coverage(sample_npe(density_estimator_npe_flexible))
coverage_control = tarp_coverage(sample_npe(density_estimator_npe_prior))

plot_coverage({**coverage_results, 'NPE (5 components),\ntrained on simulations from the prior': coverage_control})

- A mixture of five Gaussians is clearly better than a single one. A single Gaussian cannot describe our posteriors (think of the two peaks at $\pm\theta_2$). But the coverage is still significantly off.
- The estimator which was trained on simulations from the prior is the only one which is close to the diagonal. (No density estimator is perfect, and with 20 points on the curve a deviation of two to three sigma in one of them is not unusual.)

So there are two separate problems, and the second one is the training set. NPE learns the posterior for the parameter distribution it was trained on: $q(\theta|x) \propto p(x|\theta)\, p_{\rm train}(\theta)$. The training distribution takes the role of the prior, and ours puts nine times more weight on the inner cube than the prior we actually want.

In our toy model we could simply simulate a new training set, but with expensive simulations this is usually not an option. The way out is **NLE**: the likelihood $p(x|\theta)$ is a property of the simulator alone and does not depend on how the training parameters are distributed. The prior only enters afterwards, in the MCMC sampling, where we can choose it freely.

Learning the likelihood of a large data vector is difficult, though. This is where the first step pays off: we learn the likelihood of the six summary statistics instead. It does not matter that the embedding network was trained together with an imperfect posterior estimator and with the "wrong" prior. Any summary statistic gives a valid posterior. A poorly chosen one just loses information.

## The Way Out: Compression + NLE

### Use the trained embedding network to compress the NLE training Data

Careful: since we used `z_score_y='independent'`, `sbi` standardizes the data *before* passing it to our embedding network. The network that was actually trained is therefore `density_estimator_npe.embedding_net` (standardization followed by our `FCNet`), and this is what we have to use to compress the data. Applying the `FCNet` directly to the raw data would give different summaries than the ones optimized during training.

In [ ]:
summary_net = density_estimator_npe.embedding_net

with torch.inference_mode():
    x_train_nle_compressed = summary_net(x_train_nle)
    x_val_nle_compressed = summary_net(x_val_nle)
    x_test_compressed = summary_net(x_test)
    x_obs_compressed = summary_net(x_obs.unsqueeze(0))

print("Shape of the raw data:       ", x_train_nle.shape)
print("Shape of the compressed data:", x_train_nle_compressed.shape)

### What did the Embedding Network Learn?

To get a feeling for the summary statistics, we plot each of the six summaries (columns) against each of the three parameters (rows) for the test simulations. The network is free to choose any six combinations of the data, so the individual summaries have no simple meaning (and they change from one training run to the next). But you can see how they respond to the parameters:
- some summaries rise or fall steadily with $\theta_1$, others with $\theta_3$, so they can tell small from large values,
- as a function of $\theta_2$, the summaries are symmetric around zero: the data only depend on $\theta_2^2$, so no summary can know the sign of $\theta_2$,
- the dependence on $\theta_3$ is flat around zero, because the data depend on $\theta_3^3$.

In [ ]:
fig, axs = plt.subplots(3, 6, figsize=(18, 8), sharex=True, sharey='col')
for i in range(3):
    for j in range(6):
        axs[i, j].scatter(theta_test[:, i], x_test_compressed[:, j], s=1, alpha=0.5)
        axs[i, j].set_xlabel(rf'$\theta_{i+1}$')
        if i == 0:
            axs[i, j].set_title(f'Summary {j+1}')
plt.tight_layout()
plt.show()

### Now Train NLE

Here we first define the density estimator using a Neural Spline Flow (NSF) and then train it using the same custom training loop as above. Note that the roles are swapped compared to NPE: the flow now models the (compressed) data given the parameters, $p(x_{compressed}|\theta)$.

In [ ]:
# Create NSF (you can also change the build function to build_maf for comparison)
# A larger flow (e.g. num_transforms=8, num_blocks=3) reaches a slightly better loss,
# but takes much longer to train and to evaluate during the MCMC sampling.
density_estimator_nle = build_nsf(  x_train_nle_compressed,
                                    theta_train_nle,
                                    z_score_x='independent',
                                    z_score_y='independent',
                                    hidden_features=64,
                                    num_blocks=2,
                                    num_transforms=4,
                                    dropout_probability=0.,
                                    num_bins=6)

# Each batch consists of (input, condition) of the density estimator: now (compressed data, parameters)
train_loader_nle, val_loader_nle = make_loaders(x_train_nle_compressed, theta_train_nle, x_val_nle_compressed, theta_val_nle)

In [ ]:
train_density_estimator(density_estimator_nle, train_loader_nle, val_loader_nle, lr=1e-3, name='NLE')

## Create posterior sample using emcee

NLE only gives us the likelihood. To obtain the posterior, we combine it with the prior (`sbi` calls the resulting unnormalized log-posterior the *potential*) and sample it with MCMC. Here we use the ensemble sampler `emcee`. This is the step where we are free to choose the prior.

In [ ]:
# potential = log-likelihood of the (compressed) observation + log-prior
potential_fn, _ = likelihood_estimator_based_potential(density_estimator_nle, prior, x_o=x_obs_compressed)

nwalkers = 200
nsteps = 400     # number of steps per walker that we keep
discard = 1000   # burn-in
thin = 10
tot_steps = nsteps * thin + discard

def log_prob(args):
    # emcee passes the positions of many walkers at once (vectorize=True)
    theta = torch.as_tensor(args, dtype=torch.float32)
    return potential_fn(theta, track_gradients=False).numpy().astype(np.float64)

theta_0 = prior.sample((nwalkers,)).numpy()

sampler = emcee.EnsembleSampler(nwalkers, 3, log_prob, vectorize=True)
state = sampler.run_mcmc(theta_0, tot_steps, progress=True)
samples = sampler.get_chain(discard=discard, thin=thin, flat=True)

# Rough estimate of the autocorrelation time: the burn-in should be much longer than this
print("Autocorrelation time (in steps):", sampler.get_autocorr_time(discard=discard, tol=0).round(1))

# Also sample from NPE for comparison
post_NPE = DirectPosterior(density_estimator_npe, prior)
samples_npe = post_NPE.sample((100000,), x=x_obs).numpy()

## How Much Information did the Compression Lose?

In a real application we could not answer this question. For our toy model we can, because we know how the simulator works: the noise is $e^g - \mathrm{const}$ with a Gaussian $g$, so we can solve for $g$ and write down the exact likelihood of the full, uncompressed data vector (using the change of variables formula from Notebook 3). We sample the resulting exact posterior with `emcee` as well.

In [ ]:
# The exact log-posterior is implemented in tutorial_utils.py:
print(inspect.getsource(exact_log_posterior))

x_obs_np = x_obs.numpy().astype(np.float64)

# Starting points of the walkers: samples from the prior which are allowed by the exact likelihood
theta_0 = prior.sample((200000,)).numpy().astype(np.float64)
theta_0 = theta_0[np.isfinite(exact_log_posterior(theta_0, x_obs_np))][:nwalkers]

sampler_exact = emcee.EnsembleSampler(nwalkers, 3, exact_log_posterior, args=(x_obs_np,), vectorize=True)
sampler_exact.run_mcmc(theta_0, tot_steps, progress=True)
samples_exact = sampler_exact.get_chain(discard=discard, thin=thin, flat=True)

print('Standard deviations of the posteriors (theta_1, theta_2, theta_3):')
for name, s in (('Exact, full data', samples_exact), ('NLE, compressed data', samples), ('NPE', samples_npe)):
    print(f'{name:22s}', s.std(axis=0).round(3))

In [ ]:
c = ChainConsumer()
param_names = [rf"$\theta_{i+1}$" for i in range(3)]

post_df = pd.DataFrame(samples, columns=param_names)
post_npe_df = pd.DataFrame(samples_npe, columns=param_names)
post_exact_df = pd.DataFrame(samples_exact, columns=param_names)

truth_dict = dict(zip(param_names, theta_obs.tolist()))


c.add_chain(Chain(samples=post_df, color='red', name="NLE (compressed data)", sigmas=[0,1,2]))
c.add_chain(Chain(samples=post_npe_df, color='blue', name="NPE", sigmas=[0,1,2]))
c.add_chain(Chain(samples=post_exact_df, color='black', name="Exact (full data)", sigmas=[0,1,2]))
c.add_truth(Truth(location=truth_dict))

c.set_plot_config(
    PlotConfig(
        contour_label_font_size=36,
        label_font_size=18,
        tick_font_size=16,
    )
)

fig = c.plotter.plot()

What do we see?

- **NLE vs NPE:** the NLE posterior is clearly not Gaussian. It has two peaks in $\theta_2$, because the sign of $\theta_2$ cannot be determined. The single Gaussian of our NPE cannot represent this.
- **Compressed vs exact:** the exact posterior from the full data vector is narrower than the NLE posterior from the six summaries (compare the standard deviations printed above). The compression has lost information. There are several reasons: the summaries were optimized for a simple Gaussian posterior estimator, the number of training simulations is limited, and the skewed noise has a sharp lower edge which is very informative but difficult for a network to pick up.
- **Wider is not wrong.** A posterior from compressed data is a valid posterior, it just uses less information. Whether our posteriors are valid is what the coverage test in the next section checks.

Do not over-interpret a single observation: where the posterior peaks relative to the true parameters changes from one noise realization to the next.

## Coverage Test for NLE

One observation is not enough to judge a method. We now repeat the coverage test from above for NLE: is it well-calibrated even though its training data did not follow the prior?

For the coverage test, we need posterior samples for many test observations. Running `emcee` separately for each of them would take very long (hours), because every MCMC step requires an evaluation of the normalizing flow, and the cost of this is dominated by overhead rather than by the number of walkers.

Evaluating the flow for a large batch is hardly more expensive than for a small one. We therefore sample the posteriors of *all* test observations in parallel: one ensemble of walkers per observation, with a single call of the flow for all ensembles at once. `emcee` cannot do this, so we implement its algorithm (the affine-invariant "stretch move" of Goodman & Weare 2010) ourselves, which only takes a few lines:

- The ensemble is split into two halves, which are updated one after the other.
- For each walker $\theta_j$ we randomly choose a partner $\theta_k$ from the other half and propose $\theta' = \theta_k + z\,(\theta_j - \theta_k)$, with $z \in [1/a, a]$ drawn from $g(z) \propto 1/\sqrt{z}$.
- The proposal is accepted with probability $\min\left(1, z^{d-1}\, p(\theta') / p(\theta_j)\right)$, where $d$ is the number of parameters.

The next three cells contain the sampler, the sampling for all test observations, and the coverage test.

*Note: The sampling takes about one to two minutes. For smaller error bars you can increase `num_test`.*

In [ ]:
def ensemble_sampler_batched(log_prob_fn, theta_0, num_steps, a=2.0):
    """Affine-invariant ensemble sampler (stretch move) for many independent posteriors in parallel.

    log_prob_fn: maps positions [num_obs, n, num_params] to log-probabilities [num_obs, n]
    theta_0: initial positions of the walkers [num_obs, num_walkers, num_params]
    Returns the chain [num_steps, num_obs, num_walkers, num_params]
    """
    num_obs, num_walkers, num_params = theta_0.shape
    half = num_walkers // 2

    theta = theta_0.clone()
    log_p = log_prob_fn(theta)
    chain = torch.empty((num_steps, num_obs, num_walkers, num_params))

    for step in tqdm(range(num_steps)):
        # Update the two halves of each ensemble one after the other
        for active, other in [(slice(0, half), slice(half, None)), (slice(half, None), slice(0, half))]:
            current, complement = theta[:, active], theta[:, other]

            # Choose a random partner from the other half for each walker
            idx = torch.randint(complement.shape[1], current.shape[:2])
            partner = torch.gather(complement, 1, idx[..., None].expand(-1, -1, num_params))

            # Stretch move
            z = ((a - 1) * torch.rand(current.shape[:2]) + 1) ** 2 / a
            proposal = partner + z[..., None] * (current - partner)

            # Accept or reject
            log_p_proposal = log_prob_fn(proposal)
            log_ratio = (num_params - 1) * torch.log(z) + log_p_proposal - log_p[:, active]
            accept = torch.log(torch.rand(current.shape[:2])) < log_ratio

            theta[:, active] = torch.where(accept[..., None], proposal, current)
            log_p[:, active] = torch.where(accept, log_p_proposal, log_p[:, active])

        chain[step] = theta

    return chain

In [ ]:
# The same test observations as for NPE above, now compressed
x_tarp_compressed = x_test_compressed[:num_test]

# Log-posterior (up to a constant) = log-likelihood + log-prior, for all test observations at once
def log_prob_batched(theta):
    # theta: [num_test, n, num_params]
    n = theta.shape[1]
    # Repeat each compressed observation n times and add a leading sample dimension: [1, num_test * n, 6]
    x_rep = x_tarp_compressed[:, None, :].expand(-1, n, -1).reshape(1, num_test * n, -1)
    with torch.no_grad():
        log_likelihood = density_estimator_nle.log_prob(x_rep, condition=theta.reshape(num_test * n, -1))
    # The log-prior is -inf outside of the prior range
    return log_likelihood.reshape(num_test, n) + prior.log_prob(theta)

nwalkers_tarp = 50
discard_tarp = 500   # burn-in
nsteps_tarp = 500
thin_tarp = 10

theta_0 = prior.sample((num_test, nwalkers_tarp))
chain = ensemble_sampler_batched(log_prob_batched, theta_0, discard_tarp + nsteps_tarp)

# Remove burn-in and thin: [num_kept_steps, num_test, nwalkers, num_params]
chain = chain[discard_tarp::thin_tarp]

# TARP expects samples to be [num_samples, num_test, num_params]
# So we merge the steps and the walkers: [num_kept_steps * nwalkers, num_test, num_params]
posterior_samples_NLE = chain.permute(0, 2, 1, 3).reshape(-1, num_test, 3)
print("Shape of the posterior samples:", posterior_samples_NLE.shape)

In [ ]:
# Compute the coverage with TARP and compare with the NPE results from above
# (all three were trained on the same simulations, which do not follow the prior)
coverage_results['NLE (compressed data)'] = tarp_coverage(posterior_samples_NLE)
plot_coverage(coverage_results)

NLE on the compressed data follows the diagonal, although it was trained on the same simulations as the two NPE estimators. The training distribution does not matter for the likelihood, and the prior we want enters in the sampling.

(At the precision of this plot. In Notebook 8 we will look at the calibration with a more sensitive statistic and for each parameter separately, and find a mild overconfidence in $\theta_2$, the parameter with the two-peaked posterior.)

## Summary

- An embedding network trained with the NPE loss (VMIM) learns informative summary statistics of a large data vector.
- NPE uses the distribution of the training parameters as its prior. If the simulations do not follow the prior you want, its posteriors are wrong. A posterior estimator which is not flexible enough is a second, independent source of error. The coverage test reveals both.
- NLE on the compressed data avoids the problem with the prior, at the price of running an MCMC for every observation.
- Compression loses information: the result is valid, but wider than the posterior from the full data. In a real application you cannot compare with the exact posterior. Use the validation loss to compare different compression networks, and coverage tests to check the final posteriors.
- With your own training loop and sampler you control every step: optimizer, learning rate schedule, early stopping, and the MCMC.

**Next:** Notebook 8 repeats this analysis with the `sbikit` package, which provides the pipeline we just built by hand: compression, NLE, a batched ensemble sampler with convergence checks, and calibration metrics.